<a href="https://colab.research.google.com/github/Naveen-gale/deep_learning/blob/main/ppt_promte_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("ppt_prompt_dataset.csv")

In [ ]:
df.head()

,instruction,topic,audience,difficulty,slides,theme,language,presentation_type,industry,need_diagrams,need_images,need_flowcharts,need_tables,need_icons,output
0,Create a comprehensive presentation prompt on ...,Microservices Architecture,Junior Developers,Intermediate,6,Dark Mode Tech,English,Technical Training,Software Engineering,True,False,True,True,True,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
1,Design a slide deck outline for a pitch on usi...,Precision Agriculture via Drones,Commercial Farm Owners,Beginner,7,Earthy and Professional,English,Sales Pitch,Agriculture,False,True,True,True,True,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
2,Generate a workshop presentation prompt on Beh...,Behavioral Economics in Pricing,Marketing Managers,Expert,5,Corporate Minimalist,English,Workshop,Marketing,True,False,False,True,True,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
3,Create a detailed presentation outline about t...,CRISPR and Bioethics,College Students,Intermediate,8,Academic Clean,English,Academic Lecture,Biotechnology,True,True,True,False,False,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
4,Draft a presentation prompt on Zero Trust Secu...,Zero Trust Architecture,Enterprise IT Leaders,Advanced,6,Cybersecurity Dark,English,Executive Briefing,Information Technology,True,False,True,True,True,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...


In [ ]:
df.isnull().sum()

,0
instruction,0
topic,0
audience,0
difficulty,0
slides,0
theme,0
language,0
presentation_type,0
industry,0
need_diagrams,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df["input"] = (
    "Topic: " + df["topic"] +
    "\nAudience: " + df["audience"] +
    "\nDifficulty: " + df["difficulty"] +
    "\nSlides: " + df["slides"].astype(str) +
    "\nTheme: " + df["theme"] +
    "\nLanguage: " + df["language"] +
    "\nPresentation Type: " + df["presentation_type"] +
    "\nIndustry: " + df["industry"] +
    "\nNeed Diagrams: " + df["need_diagrams"].astype(str) +
    "\nNeed Images: " + df["need_images"].astype(str) +
    "\nNeed Flowcharts: " + df["need_flowcharts"].astype(str) +
    "\nNeed Tables: " + df["need_tables"].astype(str) +
    "\nNeed Icons: " + df["need_icons"].astype(str)
)

In [ ]:
train_df = df[["instruction","input","output"]]

In [ ]:
train_df

,instruction,input,output
0,Create a comprehensive presentation prompt on ...,Topic: Microservices Architecture\nAudience: J...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
1,Design a slide deck outline for a pitch on usi...,Topic: Precision Agriculture via Drones\nAudie...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
2,Generate a workshop presentation prompt on Beh...,Topic: Behavioral Economics in Pricing\nAudien...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
3,Create a detailed presentation outline about t...,Topic: CRISPR and Bioethics\nAudience: College...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...
4,Draft a presentation prompt on Zero Trust Secu...,Topic: Zero Trust Architecture\nAudience: Ente...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...


In [ ]:
train_df["text"] = (
    "<|user|>\n"
    + train_df["instruction"]
    + "\n\n"
    + train_df["input"]
    + "\n\n<|assistant|>\n"
    + train_df["output"]
)

In [ ]:
train_df.head()

,instruction,input,output,text
0,Create a comprehensive presentation prompt on ...,Topic: Microservices Architecture\nAudience: J...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...,<|user|>\nCreate a comprehensive presentation ...
1,Design a slide deck outline for a pitch on usi...,Topic: Precision Agriculture via Drones\nAudie...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...,<|user|>\nDesign a slide deck outline for a pi...
2,Generate a workshop presentation prompt on Beh...,Topic: Behavioral Economics in Pricing\nAudien...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...,<|user|>\nGenerate a workshop presentation pro...
3,Create a detailed presentation outline about t...,Topic: CRISPR and Bioethics\nAudience: College...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...,<|user|>\nCreate a detailed presentation outli...
4,Draft a presentation prompt on Zero Trust Secu...,Topic: Zero Trust Architecture\nAudience: Ente...,[PRESENTATION PROMPT]\r\n\r\n**Output Requirem...,<|user|>\nDraft a presentation prompt on Zero ...


In [ ]:
train_df[["text"]].to_json(
    "train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="train.jsonl"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=4096
    )

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
pip install peft

In [ ]:
!
!pip install "torchao>=0.16.0"
from peft import get_peft_model

model = get_peft_model(model, lora_config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=500,

    save_total_limit=2,

    fp16=True,

    report_to="none"
)

In [ ]:
pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from trl import SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,

    train_dataset=tokenized_dataset["train"],

    args=training_args
)

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


TrainOutput(global_step=3, training_loss=2.429010550181071, metrics={'train_runtime': 9.3088, 'train_samples_per_second': 1.611, 'train_steps_per_second': 0.322, 'total_flos': 29054746195200.0, 'train_loss': 2.429010550181071, 'entropy': 2.1420811149809094, 'num_tokens': 12534.0, 'mean_token_accuracy': 0.5246785912248824, 'epoch': 3.0})

In [ ]:
trainer.save_model("ppt_model")

In [ ]:
tokenizer.save_pretrained("ppt_model")

('ppt_model/tokenizer_config.json',
 'ppt_model/chat_template.jinja',
 'ppt_model/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from peft import PeftModel

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
model = PeftModel.from_pretrained(
    base_model,
    "ppt_model"
)

In [ ]:
prompt = """
Create a presentation.

Topic: Machine Learning

Audience: Beginners

Slides: 10

Theme: Modern Blue
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

In [ ]:
output = model.generate(
    **inputs,
    max_new_tokens=1500,
    temperature=0.7
)

In [ ]:
print(
    tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )
)


Create a presentation.

Topic: Machine Learning

Audience: Beginners

Slides: 10

Theme: Modern Blue
Slide 1: Introduction to Machine Learning

- What is machine learning?
- Types of ML algorithms
- Benefits and limitations of ML

Slide 2: Types of ML Algorithms

- Supervised ML
- Unsupervised ML
- Reinforcement ML

Slide 3: Supervised ML

- Goal: Predict the target variable based on input features
- Assumptions:
  - The data has been randomly sampled from a population.
  - There are no hidden variables or confounding factors.
  - Features are linearly related.
- Examples: Linear Regression, Decision Trees, Neural Networks

Slide 4: Unsupervised ML

- Goal: Discover patterns in the data without labels
- Assumptions:
  - Data is unstructured (e.g., text).
  - No clear distinction between classes.
  - No prior knowledge about the structure of the data.
- Examples: Clustering, PCA, K-means

Slide 5: Reinforcement ML

- Goal: Learn to take actions that maximize some reward function
- Assu

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained("ppt_model")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "ppt_model"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The model is now loaded and ready for use. Let's try generating a presentation for a different topic.

In [ ]:
new_prompt = """
Create a presentation.

Topic: Artificial Intelligence in Healthcare

Audience: Medical Professionals

Slides: 7

Theme: Clinical Modern
"""

new_inputs = tokenizer(new_prompt, return_tensors="pt")

new_output = model.generate(
    **new_inputs,
    max_new_tokens=1500,
    temperature=0.7
)

print(
    tokenizer.decode(
        new_output[0],
        skip_special_tokens=True
    )
)


Create a presentation.

Topic: Artificial Intelligence in Healthcare

Audience: Medical Professionals

Slides: 7

Theme: Clinical Modern
Presentation Objective: To provide an overview of how AI is being used to improve healthcare outcomes, the current limitations and challenges that it faces, as well as potential solutions for overcoming these challenges.
Slide 1: Introduction to Artificial Intelligence (AI)
- Definition and types of AI
- Examples of applications in healthcare

Slide 2: Current Challenges in Healthcare - Limitations of Current AI in Healthcare
- Limited data quality and quantity 
- Inconsistent and inaccurate diagnosis and treatment algorithms
- Difficulty in handling large volumes of patient data
- Lack of ethical guidelines for AI use in healthcare

Slide 3: AI in Healthcare: A Holistic Approach
- Use cases for AI in healthcare such as predictive analytics, drug discovery, personalized medicine, and virtual assistants
- Benefits of using AI in healthcare include imp

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Copy the saved model to Google Drive
# You can change 'My Drive/my_models' to your desired path in Google Drive
!cp -r ppt_model "/content/drive/My Drive/ppt_model_saved"